# Manual Bakery Verification and Final Classification


In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

DATA_FOLDER = Path("../data/business/interim")
VERIFICATION_FOLDER = (DATA_FOLDER / "ai_verification")
MANUAL_REVIEW_FOLDER = (VERIFICATION_FOLDER / "manual_review")
MANUAL_REVIEW_FOLDER.mkdir(parents=True, exist_ok=True)

AI_RESULTS_PATH = (VERIFICATION_FOLDER / "bakery_ai_verification_results_v5_final.csv")
REVIEW_PATH = (VERIFICATION_FOLDER / "bakery_list_review.csv")

FINAL_RESULTS_PATH = (VERIFICATION_FOLDER / "bakery_final_classification.csv")

FINAL_BAKERY_PATH = (VERIFICATION_FOLDER / "bakery_final_list.csv")

In [2]:
def update_review_file(review_df, review_path):

    if review_path.exists():
        existing_review = pd.read_excel(review_path)

        new_rows = review_df[~review_df["BakeryRank"]
                             .isin(existing_review["BakeryRank"])].copy()

        updated_review = pd.concat([existing_review, new_rows], ignore_index=True)

        updated_review = (updated_review
                          .sort_values("BakeryRank")
                          .reset_index(drop=True))

        assert not updated_review["BakeryRank"].duplicated().any()

        updated_review.to_excel(review_path, index=False)

        print(f"Existing review preserved.\nAdded {len(new_rows)} new businesses and there are now {len(updated_review)} total rows.")

    else:
        review_df.to_excel(review_path, index=False)

        print(f"Created review file with {len(review_df)} businesses.")

# Loading in AI results and review dataset

In [3]:
ai_results = pd.read_csv(AI_RESULTS_PATH)

results = ai_results.copy()

results["FinalVerdict"] = results["AIVerdict"]
results["ReviewNote"] = ""
results["Reviewed"] = False

results = (
    results
    .sort_values("BakeryRank")
    .reset_index(drop=True)
)

print(f"AI verification results loaded: {len(results)}")

AI verification results loaded: 28847


In [4]:
print(f"AI verdicts: \n{results["AIVerdict"].value_counts()}")
print(f"\nCurrent final verdicts: \n{results["FinalVerdict"].value_counts()}")
print(f"\nFHRS business types: \n{results["BusinessType"].value_counts()}")

AI verdicts: 
AIVerdict
NOT_BAKERY    19875
UNCLEAR        6029
BAKERY         2943
Name: count, dtype: int64

Current final verdicts: 
FinalVerdict
NOT_BAKERY    19875
UNCLEAR        6029
BAKERY         2943
Name: count, dtype: int64

FHRS business types: 
BusinessType
Restaurant/Cafe/Canteen                  7578
Retailers - other                        5860
Other catering premises                  4766
Caring Premises                          2445
Takeaway/sandwich shop                   1880
School/college/university                1824
Pub/bar/nightclub                        1298
Mobile caterer                           1101
Manufacturers/packers                     702
Hotel/bed & breakfast/guest house         565
Distributors/Transporters                 369
Retailers - supermarkets/hypermarkets     271
Importers/Exporters                       174
Farmers/growers                            14
Name: count, dtype: int64


# Checking for inactive businesses

In [5]:
results["InactiveFlag"] = (results["AIReason"]
                           .fillna("")
                           .str.startswith("INACTIVE:"))

inactive_results = results[results["InactiveFlag"]].copy()

print(f"Businesses flagged as inactive: {len(inactive_results)}")

print(pd.crosstab(results["AIVerdict"], results["InactiveFlag"]))

Businesses flagged as inactive: 166
InactiveFlag  False  True 
AIVerdict                 
BAKERY         2936      7
NOT_BAKERY    19724    151
UNCLEAR        6021      8


In [6]:
inactive_to_change = inactive_results[inactive_results["AIVerdict"] != "NOT_BAKERY"].copy()

display(inactive_to_change[[
    "BusinessName",
    "BusinessType",
    "AIVerdict",
    "BakeryRank",
    "AIReason"
    ]])

,BusinessName,BusinessType,AIVerdict,BakeryRank,AIReason
619,Roll N Bake Ltd,Other catering premises,UNCLEAR,620,"INACTIVE: Companies House shows ""Roll N Bake Ltd"" is in the process of voluntary strike-off."
2010,Happi Cheesecakes,Other catering premises,BAKERY,2011,INACTIVE: Website states it is undergoing rebrand but previously made cheesecakes.
2446,cake cult london,Restaurant/Cafe/Canteen,BAKERY,2447,"INACTIVE: Cake Cult London was a vegan bakery offering plant-based cakes, pastries, brownies, and cookies."
2635,Karen's Cake House,Other catering premises,BAKERY,2636,INACTIVE: Karen's Cake House specialized in baking and decorating bespoke cakes for special occasions.
3333,Coughlans,Retailers - other,BAKERY,3334,"INACTIVE: News reports and business listings confirm it was a bakery chain known for breads, cakes, and pastries."
3550,Lidia's Sweet Kitchen,Restaurant/Cafe/Canteen,UNCLEAR,3551,"INACTIVE: Company is dissolved, and past activity is unclear."
3638,Akimalans Ltd,Retailers - other,UNCLEAR,3639,INACTIVE: Akimalans Ltd is dissolved and its business activity is not specified.
3838,KYD Trading Ltd,Restaurant/Cafe/Canteen,UNCLEAR,3839,INACTIVE: KYD Trading Ltd is dissolved and its business activity is not specified.
4216,Percy Ingle,Retailers - other,BAKERY,4217,INACTIVE: It was a retail bakery chain that closed all its stores in 2020.
4280,Twins Wedding Shop,Retailers - other,BAKERY,4281,INACTIVE: Yell and news articles confirm it was a wedding cake shop that closed in 2024.


# Parsing for bakery-related words

In [7]:
bakery_words = ["bakery", "baker", "bake", "bakes", "cake", "cakes", "bread", "pastry", "patisserie", "cookie", "cookies", "cupcake", "sourdough", "doughnut", "donut", "bagel", "baklava", "pie",
                 # added after inspection
                "biscuit", "brownie", "muffin", "croissant", "focaccia", "brioche", "pretzel", "gingerbread", "macaron"]


bakery_word_pattern = "|".join(bakery_words)


results["BakeryWordInName"] = (results["BusinessName"]
                               .fillna("")
                               .str.lower()
                               .str.contains(bakery_word_pattern))


print(f"Businesses with bakery-related words in their name: {results["BakeryWordInName"].sum()}")

Businesses with bakery-related words in their name: 2799


In [8]:
bakery_word_unclear = results[(results["BakeryWordInName"]) & (results["AIVerdict"] == "UNCLEAR")].copy()

bakery_word_not_bakery = results[(results["BakeryWordInName"]) & (results["AIVerdict"] == "NOT_BAKERY")].copy()

bakery_word_bakery = results[(results["BakeryWordInName"]) & (results["AIVerdict"] == "BAKERY")].copy()

print(f"Bakery-word names currently BAKERY: {len(bakery_word_bakery)}")

print(f"Bakery-word names currently UNCLEAR: {len(bakery_word_unclear)}")

print(f"Bakery-word names currently NOT_BAKERY: {len(bakery_word_not_bakery)}")

Bakery-word names currently BAKERY: 1744
Bakery-word names currently UNCLEAR: 832
Bakery-word names currently NOT_BAKERY: 223


# Reviewing bakery-related UNCLEAR businesses

In [20]:
bakery_word_unclear["BakeryInName"] = (bakery_word_unclear["BusinessName"]
                                       .fillna("")
                                       .str.lower()
                                       .str.contains("bakery", regex=False))


unclear_with_bakery_name = bakery_word_unclear[bakery_word_unclear["BakeryInName"]].copy()


print(f'UNCLEAR businesses with "bakery" in the name: {len(unclear_with_bakery_name)}')

UNCLEAR businesses with "bakery" in the name: 125


In [10]:
unclear_bakery_review = (
    unclear_with_bakery_name[[
        "BusinessNameClean",
        "BusinessName",
        "BusinessType",
        "Address",
        "PostCode",
        "LocalAuthorityName",
        "BakeryRank",
        "BakeryScore",
        "AIVerdict",
        "AIReason"
        ]].sort_values("BakeryRank").copy())

# Empty columns for me to fill in manually
unclear_bakery_review["ManualVerdict"] = ""
unclear_bakery_review["ReviewNote"] = ""

UNCLEAR_BAKERY_REVIEW_PATH = (MANUAL_REVIEW_FOLDER / "01_unclear_bakery_name_review.xlsx")

update_review_file(unclear_bakery_review, UNCLEAR_BAKERY_REVIEW_PATH)

Existing review preserved.
Added 0 new businesses and there are now 125 total rows.


# Reviewing bakery-related NOT_BAKERY businesses

In [11]:
not_bakery_bakery_words = results[(results["AIVerdict"] == "NOT_BAKERY")
                                   & (results["BakeryWordInName"]) 
                                   & (~results["InactiveFlag"])].copy()

print(f"Bakery-word businesses currently NOT_BAKERY: {len(not_bakery_bakery_words)}")

Bakery-word businesses currently NOT_BAKERY: 222


In [12]:
not_bakery_review = (not_bakery_bakery_words[[
    "BusinessNameClean",
    "BusinessName",
    "BusinessType",
    "Address",
    "PostCode",
    "LocalAuthorityName",
    "BakeryRank",
    "BakeryScore",
    "AIVerdict",
    "AIReason"
    ]].sort_values("BakeryRank").copy())

not_bakery_review["ManualVerdict"] = ""
not_bakery_review["ReviewNote"] = ""

NOT_BAKERY_REVIEW_PATH = (MANUAL_REVIEW_FOLDER / "02_not_bakery_bakery_words_review.xlsx")

update_review_file(not_bakery_review, NOT_BAKERY_REVIEW_PATH)

Existing review preserved.
Added 27 new businesses and there are now 222 total rows.


# Applying completed manual review

In [21]:
# Rebuild the current review state safely
results = pd.read_csv(AI_RESULTS_PATH)

results["FinalVerdict"] = results["AIVerdict"]
results["ReviewNote"] = ""
results["Reviewed"] = False


# Apply the inactive-business rule first
results["InactiveFlag"] = (results["AIReason"]
                           .fillna("")
                           .str.startswith("INACTIVE:"))

results.loc[results["InactiveFlag"], "FinalVerdict"] = "NOT_BAKERY"
results.loc[results["InactiveFlag"], "ReviewNote"] = ("Excluded because AI verification identified the business as inactive, closed or dissolved.")

def apply_manual_review(review_path):

    review = pd.read_excel(review_path)

    review["ManualVerdict"] = (review["ManualVerdict"]
                               .fillna("")
                               .str.strip()
                               .str.upper()
                               .str.replace(" ", "_", regex=False))

    review["ReviewNote"] = (review["ReviewNote"].fillna(""))

    # Only import rows that actually contain a completed decision
    review = review[review["ManualVerdict"].isin(["BAKERY", "NOT_BAKERY", "INACTIVE"])].copy()

    # INACTIVE means excluded from current bakery provision
    review.loc[review["ManualVerdict"] == "INACTIVE", "ManualVerdict"] = "NOT_BAKERY"

    # BakeryRank is unique and avoids problems if a name was edited in Excel
    review_updates = review.set_index("BakeryRank")

    reviewed_rows = results["BakeryRank"].isin(review_updates.index)

    results.loc[reviewed_rows, "FinalVerdict"] = (results.loc[reviewed_rows, "BakeryRank"]
                                                  .map(review_updates["ManualVerdict"]))

    results.loc[reviewed_rows, "ReviewNote"] = (results.loc[reviewed_rows, "BakeryRank"]
                                                .map(review_updates["ReviewNote"]))

    results.loc[reviewed_rows, "Reviewed"] = True

    return len(review), reviewed_rows.sum()


review_01_count, review_01_matched = apply_manual_review(UNCLEAR_BAKERY_REVIEW_PATH)

review_02_count, review_02_matched = apply_manual_review(NOT_BAKERY_REVIEW_PATH)


# Final safety checks before anything gets saved
blank_verdicts = (results["FinalVerdict"].fillna("").eq("").sum())

valid_verdicts = results["FinalVerdict"].isin(["BAKERY", "NOT_BAKERY", "UNCLEAR"])

print(f"01 review applied: {review_01_matched}/{review_01_count}")
print(f"02 review applied: {review_02_matched}/{review_02_count}")
print(f"Total manually reviewed: {results['Reviewed'].sum()}")
print(f"Blank FinalVerdict rows: {blank_verdicts}")

print(f"\nFinal verdicts:\n{results["FinalVerdict"].value_counts()}")


assert review_01_matched == review_01_count
assert review_02_matched == review_02_count
assert blank_verdicts == 0
assert valid_verdicts.all()


results = (results.sort_values("BakeryRank").reset_index(drop=True))

01 review applied: 125/125
02 review applied: 195/195
Total manually reviewed: 320
Blank FinalVerdict rows: 0

Final verdicts:
FinalVerdict
NOT_BAKERY    19978
UNCLEAR        5896
BAKERY         2973
Name: count, dtype: int64


In [23]:
review_01 = pd.read_excel(UNCLEAR_BAKERY_REVIEW_PATH)
review_02 = pd.read_excel(NOT_BAKERY_REVIEW_PATH)

pending_01 = (review_01["ManualVerdict"]
              .fillna("")
              .str.strip()
              .eq("").
              sum())

pending_02 = (review_02["ManualVerdict"]
              .fillna("")
              .str.strip()
              .eq("")
              .sum())

print(f"01 still to review: {pending_01}")
print(f"02 still to review: {pending_02}")

01 still to review: 0
02 still to review: 27


# Reviewing BAKERY classifications

In [14]:
bakeries_to_review = results[(results["FinalVerdict"] == "BAKERY") 
                             & (~results["Reviewed"])].copy()

print(f"Bakeries still to manually review: {len(bakeries_to_review)}")

Bakeries still to manually review: 2936


In [15]:
review_reason_words = [
    # Weak evidence / mostly inferred from the name or FHRS listing
    "name suggests",
    "name indicates",
    "business name",
    "likely",
    "imply",
    "no conflicting evidence",
    "food and beverage business",

    # Wording suggesting another activity may be primary
    "primarily a restaurant",
    "primarily a cafe",
    "primarily a café",
    "primarily a takeaway",
    "primarily a dessert",
    "dessert shop",
    "dessert spot",
    "catering service",

    # Bakery products may only be incidental
    "bakery section",
    "incidental",
    "limited bakery",
    "not a bakery",
    "not traditional bakery",
    "not baked goods",
    "not bakery products",

    # Institutional / non-commercial activity
    "baking club",
    "baking workshop",
    "after-school",
    "cake decorating supplies",
    "supplies and equipment"
]

review_reason_pattern = "|".join(review_reason_words)

bakery_reasoned_review = bakeries_to_review[bakeries_to_review["AIReason"]
                                            .fillna("")
                                            .str.lower()
                                            .str.contains(review_reason_pattern)].copy()

print(f"Bakeries flagged for closer review: {len(bakery_reasoned_review)}")

Bakeries flagged for closer review: 92


In [16]:
unusual_bakery_types = [
    "Caring Premises",
    "School/college/university",
    "Retailers - supermarkets/hypermarkets",
    "Distributors/Transporters",
    "Hotel/bed & breakfast/guest house",
    "Importers/Exporters",
    "Farmers/growers",
    "Pub/bar/nightclub"]

bakery_type_review = bakeries_to_review[bakeries_to_review["BusinessType"].isin(unusual_bakery_types)].copy()

bakery_reasoned_review = pd.concat([bakery_reasoned_review,bakery_type_review]
                                   ).drop_duplicates(subset="BakeryRank")

print(f"Total priority BAKERY review: {len(bakery_reasoned_review)}")

Total priority BAKERY review: 111


In [17]:
bakery_reasoned_review = (
    bakery_reasoned_review[[
        "BusinessNameClean",
        "BusinessName",
        "BusinessType",
        "Address",
        "PostCode",
        "LocalAuthorityName",
        "BakeryRank",
        "BakeryScore",
        "AIVerdict",
        "AIReason"
        ]].sort_values("BakeryRank").copy())

bakery_reasoned_review["ManualVerdict"] = "BAKERY"
bakery_reasoned_review["Checked"] = ""
bakery_reasoned_review["ReviewNote"] = ""

BAKERY_REASONED_REVIEW_PATH = (MANUAL_REVIEW_FOLDER / "03_bakery_reason_review.xlsx")

update_review_file(bakery_reasoned_review, BAKERY_REASONED_REVIEW_PATH)

Existing review preserved.
Added 12 new businesses and there are now 111 total rows.


# Importing completed priority BAKERY review

In [18]:
completed_bakery_review = pd.read_excel(BAKERY_REASONED_REVIEW_PATH)

# Only use rows that I actually checked
completed_bakery_review["Checked"] = (completed_bakery_review["Checked"]
                                      .fillna("")
                                      .astype(str)
                                      .str.strip()
                                      .str.upper())

completed_bakery_review = completed_bakery_review[completed_bakery_review["Checked"] == "X"].copy()

# Clean manual decisions
completed_bakery_review["ManualVerdict"] = (completed_bakery_review["ManualVerdict"]
                                            .fillna("")
                                            .str.strip()
                                            .str.upper()
                                            .str.replace(" ", "_", regex=False))

completed_bakery_review["ReviewNote"] = (completed_bakery_review["ReviewNote"].fillna(""))

# Check that every reviewed row has a valid decision
valid_manual_verdicts = ["BAKERY", "NOT_BAKERY", "INACTIVE"]
invalid_manual_verdicts = (~completed_bakery_review["ManualVerdict"].isin(valid_manual_verdicts))

assert invalid_manual_verdicts.sum() == 0

# INACTIVE businesses are excluded from current provision
completed_bakery_review.loc[completed_bakery_review["ManualVerdict"] == "INACTIVE", "ManualVerdict"] = "NOT_BAKERY"

# Apply the completed review
review_03_updates = (completed_bakery_review.set_index("BakeryRank"))
review_03_rows = (results["BakeryRank"].isin(review_03_updates.index))

assert review_03_rows.sum() == len(completed_bakery_review)


results.loc[review_03_rows, "FinalVerdict"] = (results.loc[review_03_rows, "BakeryRank"]
                                               .map(review_03_updates["ManualVerdict"]))

results.loc[review_03_rows, "ReviewNote"] = (results.loc[review_03_rows, "BakeryRank"]
                                             .map(review_03_updates["ReviewNote"]))

results.loc[review_03_rows, "Reviewed"] = True

# Final checks
blank_verdicts = (results["FinalVerdict"]
                  .fillna("")
                  .eq("")
                  .sum())

valid_final_verdicts = (results["FinalVerdict"].isin(["BAKERY", "NOT_BAKERY", "UNCLEAR"]))

In [19]:
print(f"03 review applied: {review_03_rows.sum()}/{len(completed_bakery_review)}")

print(f"Total manually reviewed: {results['Reviewed'].sum()}")

print(f"Blank FinalVerdict rows: {blank_verdicts}")

print(f"\nFinal verdicts:\n{results['FinalVerdict'].value_counts()}")

assert blank_verdicts == 0
assert valid_final_verdicts.all()


# Save current review state
results = (results
           .sort_values("BakeryRank")
           .reset_index(drop=True))

results.to_csv(REVIEW_PATH, index=False)

print("\nAll completed manual reviews saved.")

03 review applied: 99/99
Total manually reviewed: 419
Blank FinalVerdict rows: 0

Final verdicts:
FinalVerdict
NOT_BAKERY    20029
UNCLEAR        5896
BAKERY         2922
Name: count, dtype: int64

All completed manual reviews saved.


In [24]:
review_03 = pd.read_excel(BAKERY_REASONED_REVIEW_PATH)

pending_03 = (~review_03["Checked"]
              .fillna("")
              .astype(str)
              .str.strip()
              .str.upper()
              .eq("X")).sum()

print(f"03 still to review: {pending_03}")

03 still to review: 12
